# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0678/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh / Content Opportunity Scoring. I will treat it as a ranking task because the goal is to rank pages from highest to lowest priority for review. The output is meant to help a content team decide which pages to look at first.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "ranking"
lane = "Refresh / Content Opportunity Scoring"

print("Lane:", lane)
print("Task type:", task_type)

Lane: Refresh / Content Opportunity Scoring
Task type: ranking


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter exercise, I will use trend_direction == "down" as a proxy target. I will create a label called is_declining_label, where 1 means the page's observed trend is down and 0 means it is not down.

This is an observed current-window proxy, not a future outcome or causal target.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target_definition = 'trend_direction == "down"'

print("Proxy target:", target_definition)

Proxy target: trend_direction == "down"


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use Precision@50. A good result means that a large share of the top 50 pages ranked for review match the positive proxy label. This fits the decision because the team has limited time and will likely review the highest-ranked pages first.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
k = 50

print(f"Success metric: Precision@{k}")
print("Higher is better.")

Success metric: Precision@50
Higher is better.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item/page.

I will use content_id as the page-level identifier. Each row represents one content item and contains its observed performance and content signals. I will also create the is_declining_label proxy target from trend_direction.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

lane_df = (
    df[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

lane_df["is_declining_label"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

print("Rows:", len(lane_df))
print("One row = one content item/page")

display(lane_df.head())

display(
    lane_df[
        ["content_id", "trend_direction", "is_declining_label"]
    ].head(10)
)

Rows: 30000
One row = one content item/page


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### 5. Why ML beats a fixed rule here

A fixed rule can identify simple cases, such as old pages or pages with declining trends. However, page performance depends on several signals at the same time, including visibility, freshness, position, demand, and engagement.

ML is worth testing because it can learn combinations of these signals and produce a more useful ranking than one simple rule. The ML result should still be compared with a transparent fixed-rule baseline.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signals = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction",
    "ctr",
    "avg_position",
    "word_count",
    "engagement_rate"
]

available_signals = [col for col in signals if col in lane_df.columns]

print("Available signals:")
print(available_signals)

print("\nML is worth testing because multiple signals can interact.")

Available signals:
['impressions_90d', 'sessions_90d', 'content_age_days', 'trend_direction', 'ctr', 'avg_position', 'word_count', 'engagement_rate']

ML is worth testing because multiple signals can interact.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.